In [1]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ubuntu/miniconda/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.5.1 with CUDA 1201 (you have 2.8.0+cu128)
    Python  3.11.10 (you have 3.11.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
max_seq_length = None
load_in_4bit = True
dtype = None 
model_name = '/data2/finetuned_llms/end_to_end_v2_phonemes_only'

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

==((====))==  Unsloth 2025.8.1: Fast Llama patching. Transformers: 4.55.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.278 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.8.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [9]:

from datasets import load_dataset

base_path = '/data2/jsonl/val.jsonl'

data_files = {
    'val': base_path
}

dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset['val']['text'][0])


<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity to help restore communication for a paralyzed patient. For each 80 ms time bin of neural activity, a model trained with CTC loss provides the 10 most probable ARPAbet phonemes, listed from most probable to least probable. The special token "~" denotes the CTC blank and should never be included in the response, and "<>" denotes a space and should be converted to a single space. The decoded outputs are listed on separate lines ordered by time. CTC post-processing has already removed outputs where the CTC blank was most probable as well as outputs with consecutive repeats not separated by a CTC blank. Produce fluent, standard spelling and grammar. Output only the final sentence text, with no explanations or metadata.

DH ~ SH K S T JH L CH G
EH ~ AE IH EY AH IY AY S AA
AA ~ AE AH K AO EH S AY IH
K ~ AA S AH AE G EH AO L
AH ~ K R T S W ER EH G
S ~ AH Z T R IY SH ER K
<> ~ S Z T D IY K AH IH
R ~ K W

In [11]:
batch_size = 1
split = "val"

last_lines_val = []
n = len(dataset[split])

for batch_idx in range(880):
    
    if batch_idx % 100 == 0:
        print(batch_idx)

    # Grab a batch of texts
    batch = dataset[split][batch_idx]
    val_texts = batch["text"]
    
    # Tokenize the batch
    inputs = tokenizer(
        val_texts,
        return_tensors='pt',
        padding=True,
        truncation=True
    ).to('cuda')

    # Generate
    outputs = model.generate(
        **inputs,
        use_cache=True, 
        max_new_tokens=400
    )

    # Decode all outputs
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Keep just the last non-empty line from each sequence
    for seq in decoded:
        lines = [l.strip() for l in seq.splitlines() if l.strip()]
        last_line = lines[-1] if lines else ""
        last_lines_val.append(last_line)

0
100
200
300
400
500
600
700
800


In [14]:
import json
from cer_wer import _cer_and_wer
with open("/data2/jsonl/val_ground_truth.json", "r") as f:
    val_gt = json.load(f)
    
_cer_and_wer(last_lines_val, val_gt)

(np.float64(0.1903032504085709), np.float64(0.2624113475177305))